# SEER GBM 数据清洗

数据来源: SEER Research Data, 17 Registries, Nov 2025 Sub (2000-2023)

筛选条件 (SEER*Stat Query):
- Primary Site: C71.0-C71.9 (Brain)
- ICD-O-3 Hist/behav: 9440/3 (Glioblastoma NOS), 9441/3 (Giant cell glioblastoma), 9445/3 (Glioblastoma, IDH-mutant)
- Year of diagnosis: 2004-2019

变量说明:
- Age recode with <1 year olds and 90+: 诊断年龄分组 (00 years ~ 90+ years)
- Race recode (W, B, AI, API): 种族 (White/Black/American Indian/Asian or Pacific Islander/Unknown)
- Sex: 性别 (Male/Female)
- Year of diagnosis: 诊断年份 (2004-2019)
- Primary Site - labeled: 原发部位 (ICD-O-3 topography)
- ICD-O-3 Hist/behav, malignant: 组织学类型 (ICD-O-3 morphology)
- Grade Recode (thru 2017): 分化程度 (Grade I~IV + Unknown)
- Summary stage 2000 (1998-2017): 总分期 (Localized/Regional/Distant/Unknown)
- RX Summ--Surg Prim Site (1998-2022): 手术 (0=No, 其他=Yes)
- Radiation recode: 放疗 (Beam radiation/Radioactive implants/Radioisotopes/None等)
- Chemotherapy recode (yes, no/unk): 化疗 (Yes/No/Unknown)
- Vital status recode (study cutoff used): 生存状态 (Alive/Dead)
- Survival months: 生存月数 (0~9999, 9999=unknown)

排除标准:
1. 年龄 <20岁 (儿童GBM罕见，本研究仅纳入成人)
2. 种族 Unknown (缺失信息)
3. Survival months 为 NA 或 ≤0 (不可用的生存时间)

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('data.csv')
print(df.columns.to_list())
print(df['Summary stage 2000 (1998-2017)'].value_counts())

In [ ]:
# Age recode: 原始分组为5岁间隔，合并为3组
# <20岁排除 (儿童GBM罕见，本研究仅纳入成人)
age_map = {
    '00 years': 'exclude',
    '01-04 years': 'exclude',
    '05-09 years': 'exclude',
    '10-14 years': 'exclude',
    '15-19 years': 'exclude',
    '20-24 years': '<60',
    '25-29 years': '<60',
    '30-34 years': '<60',
    '35-39 years': '<60',
    '40-44 years': '<60',
    '45-49 years': '<60',
    '50-54 years': '<60',
    '55-59 years': '<60',
    '60-64 years': '60-69',
    '65-69 years': '60-69',
    '70-74 years': '≥70',
    '75-79 years': '≥70',
    '80-84 years': '≥70',
    '85-89 years': '≥70',
    '90+ years': '≥70'
}
df['Age Group'] = df['Age recode with <1 year olds and 90+'].map(age_map)
df = df[df['Age Group'] != 'exclude']
print(df['Age Group'].unique())

In [ ]:
# Race recode: 排除 Unknown，仅保留有明确种族记录的患者
df['Race'] = df['Race recode (W, B, AI, API)']
df = df[df['Race'] != 'Unknown']
print(df['Race'].unique())

In [ ]:
# Sex: 编码为 0/1 便于 Cox 回归建模
df['Sex'] = df['Sex'].map({'Male': 0, 'Female': 1})
print(df['Sex'].unique())

In [ ]:
# Grade Recode: Blank(s) 转为 Unknown (SEER中 Blank = 缺失)
df['Grade'] = df['Grade Recode (thru 2017)'].replace('Blank(s)', 'Unknown')
print(df['Grade'].unique())

In [ ]:
# Summary stage 2000: Blank(s) 转为 Unknown/unstaged
df['Stage'] = df['Summary stage 2000 (1998-2017)'].replace('Blank(s)', 'Unknown/unstaged')
print(df['Stage'].unique())

In [ ]:
# RX Summ--Surg Prim Site: 0=无手术, 其他编码=有手术 → 二值化
df['Surgery'] = (df['RX Summ--Surg Prim Site (1998-2022)'] != 0).astype(int)
print(df['Surgery'].value_counts())

In [ ]:
# Radiation recode: 含多种放疗方式，均视为"接受放疗"
# None/已排除的编码 = 未放疗
radiation_yes = [
    'Beam radiation',
    'Radiation, NOS method or source not specified',
    'Combination of beam with implants or isotopes',
    'Radioactive implants (includes brachytherapy) (1988+)',
    'Radioisotopes (1988+)'
]
df['Radiation'] = df['Radiation recode'].isin(radiation_yes).astype(int)
print(df['Radiation'].value_counts())

In [ ]:
# Chemotherapy recode: Yes=1, No/Unknown=0
# No和Unknown合并处理 (SEER不区分"未化疗"和"未知")
df['Chemotherapy'] = df['Chemotherapy recode (yes, no/unk)'].map({'Yes': 1, 'No/Unknown': 0})
print(df['Chemotherapy'].value_counts())

In [ ]:
# Vital status recode: Alive=0, Dead=1
# 注意: lifelines要求 event=1 表示事件发生(死亡), 0=删失
# SEER编码: Alive=存活 → 删失(0), Dead=死亡 → 事件(1)
df['Vital status recode'] = df['Vital status recode (study cutoff used)'].map({'Alive': 0, 'Dead': 1})
print(df['Vital status recode'].value_counts())

In [ ]:
# Survival months: 转为数值型，排除 NA 和 ≤0 的记录
df['Survival months'] = pd.to_numeric(df['Survival months'], errors='coerce')
df = df[df['Survival months'].notna()]
df = df[df['Survival months'] > 0]
print(df['Survival months'].info())

In [ ]:
print(df.info())

In [ ]:
# 保留分析所需变量，导出清洗后数据
df = df[['Age Group', 'Race', 'Sex', 'Year of diagnosis', 
         'Primary Site - labeled', 'ICD-O-3 Hist/behav, malignant',
         'Grade', 'Stage', 'Surgery', 'Radiation', 'Chemotherapy',
         'Vital status recode', 'Survival months']]
df.to_csv('cleaned_data.csv', index=False)